___
# <span style="color:darkgreen">📊 Retail Analytics: Walmart Case Study</span>
___

### Step 2. Data Pre-Processing

**Objective**: Prepare a refined dataset optimized for analysis and modeling tasks

**Action Items**   
1. Handling missing values
2. Creating calculated columns showing number of days taken for shipment 
3. Exporting Cleaned Dataset 


In [1]:
#Importing pandas libraries
import pandas as pd
import numpy as np
#Display setting for pandas
pd.set_option('display.max_columns', None)              #show all columns
pd.set_option('display.float_format','{:.2f}'.format)    #format all decimals to 2 places

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
#Load the dataset
df = pd.read_csv('../data/Walmart Retail Data.csv', encoding='latin-1')
# Display dataset dimensions
print("Dataset loaded successfully!")
print("="*70)
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dataset loaded successfully!
Dataset Shape: 8,399 rows × 25 columns


#### 1. Handling missing values

Identified 966 records have missing values as per below   
 - Customer's age is missing for 903 records
 - Product base margine is mnissing for 63 records

***Strategies to be followed to find impute missing values***

##### 1.1 Missing age of customers

- 903 out of 8399 (~10.75%) is a moderate proportion. It’s not negligible, so required to impute age wherever missing
- Calculate Standard deviation of age
- Calculate overall average age
- Calculate group based average 

In [10]:
# Calculating standard deviation of customer's age
std_age_population = df['Customer Age'].dropna().std(ddof=0)
std_age_population = round (std_age_population,2)
print("Standard deviation of age:", std_age_population)

# Calculating overall average of customer's age
df['Customer Age'] = pd.to_numeric(df['Customer Age'], errors='coerce')
# Drop missing values before calculating mean
average_age = df['Customer Age'].dropna().mean()
average_age = round(average_age, 2)
print("Overall Average of Customer Age:", average_age)

# Calculating average of customer's age by 'Customer Segment' level
group_avg_age = df.groupby('Customer Segment')['Customer Age'].mean()
print(group_avg_age)


Standard deviation of age: 9.52
Overall Average of Customer Age: 54.54
Customer Segment
Consumer         53.94
Corporate        54.71
Home Office      54.76
Small Business   54.56
Name: Customer Age, dtype: float64


**Conclusion**  
The age of customers vary about 9.5 years from the average. Calculated average by Customer segment as well as overall level. Since averge value of agent is almost same by segment level and overall level, proceeding imputing missing ages with overall average **54.54**


##### 1.2 Missing Product base Margin

- Product Base Margin has 63 missing values(0.75%), this will be imputed with average value of 'Product Sub-Category'
- Calculate averge value of product base margin based on the 'Product Sub-Category'

In [12]:
# Calculating average of Product base margin by 'Product-Sub category' level
avg_margin_by_subcategory = df.groupby('Product Sub-Category')['Product Base Margin'].mean().round(2)
print(avg_margin_by_subcategory)


Product Sub-Category
Appliances                       0.55
Binders and Binder Accessories   0.37
Bookcases                        0.66
Chairs & Chairmats               0.63
Computer Peripherals             0.59
Copiers and Fax                  0.42
Envelopes                        0.37
Labels                           0.38
Office Furnishings               0.53
Office Machines                  0.44
Paper                            0.37
Pens & Art Supplies              0.53
Rubber Bands                     0.53
Scissors, Rulers and Trimmers    0.64
Storage & Organization           0.67
Tables                           0.69
Telephones and Communication     0.58
Name: Product Base Margin, dtype: float64


#### 2. Imputing missing values

##### 2.1 Imputing Missing age of customers

In [21]:
# Impute missing values with the average_age
df['Customer Age'] = pd.to_numeric(df['Customer Age'], errors='coerce')
df['Customer Age'] = df['Customer Age'].fillna(average_age)

# Verify that there are no missing values left
missing_count = df['Customer Age'].isna().sum()
print(f"Number customers with age missing : {missing_count}")


Number customers with age missing : 0


##### 2.2 Imputing Missing Product base margin based on corresponding Product Sub-catogory's average value

In [22]:
# Impute missing values with the average of Product base margin by product sub-subcategory level
df['Product Base Margin'] = df['Product Base Margin'].fillna(
    df.groupby('Product Sub-Category')['Product Base Margin'].transform('mean')
)
# Verify that there are no missing values left
missing_margin = df['Product Base Margin'].isna().sum()
print(f"Number of records with Product Base Margin missing : {missing_margin}")


Number of records with Product Base Margin missing : 0


#### 2. Creating calculated columns showing shipment delays

This is to calculate delays of shipment based on the oder date and shipment date provided in the data

In [27]:
# Calculate shipment delays based on order date and shipment date
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y', errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y', errors='coerce')

# Calculate difference in days
df['Shipment Delays'] = (df['Ship Date'] - df['Order Date']).dt.days

# Preview the result
print(df[['Order Date', 'Ship Date', 'Shipment Delays']].head())


  Order Date  Ship Date  Shipment Delays
0 2012-01-01 2012-01-02                1
1 2012-01-01 2012-01-03                2
2 2012-01-02 2012-01-02                0
3 2012-01-02 2012-01-02                0
4 2012-01-02 2012-01-04                2


#### 3. Exporting Cleaned Database

In [28]:
# Export DataFrame to CSV file
df.to_csv("Cleaned_Walmart Retail Data.csv", index=False)

print("Data exported successfully to Cleaned_Walmart Retail Data.csv")


Data exported successfully to Cleaned_Walmart Retail Data.csv
